## Template weather data extractor

In [ ]:
"""Extract: ask the source for data and keep an untouched Bronze copy."""

import json
from datetime import UTC, date, datetime
from pathlib import Path
from uuid import uuid4

import httpx

In [ ]:
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
DAILY_FIELDS = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"]

In [ ]:
class TemporarySourceError(RuntimeError):
    """The source failed in a way that might work if we try again."""

In [ ]:
def fetch_weather(latitude: float, longitude: float, start: date, end: date) -> dict:
    """Download daily weather for one place, from start to end (both included).

    Raises:
        TemporarySourceError: Network trouble, rate limiting or a server error.
        httpx.HTTPStatusError: Any other bad answer, such as a wrong parameter.
    """
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start.isoformat(),
        "end_date": end.isoformat(),
        "daily": ",".join(DAILY_FIELDS),
        "timezone": "UTC",
    }
    try:
        response = httpx.get(ARCHIVE_URL, params=params, timeout=30)
        print(response.text)
        print(response.status_code)
        print(response.headers)
        print(response.json())
        print(response.request.url)
        print(response.request.headers)
        # print(response.request.content)
        # print(response.request.body)
        # print(response.request.read())
        # print(response.request.stream())
        # print(response.request.is_closed)
        # print(response.request.is_stream_consumed)
        print(response.url)
    except httpx.TransportError as error:
        raise TemporarySourceError(f"Could not reach Open-Meteo: {error}") from error
    if response.status_code == 429 or response.status_code >= 500:
        raise TemporarySourceError(f"Open-Meteo answered {response.status_code}. Try later.")
    response.raise_for_status()
    return response.json()

In [ ]:
payload = fetch_weather(
    *(5.56, -0.20),  # *config.LOCATIONS["accra"]
    date(2026, 11, 1),
    date(2026, 11, 3),
)

In [ ]:
payload

In [ ]:
def save_bronze(payload: dict, location: str, bronze_dir: str) -> Path:
    """Save the response exactly as received, plus metadata about where it came from."""
    now = datetime.now(UTC)
    folder = Path(bronze_dir) / f"ingest_date={now.date()}" / f"location={location}"
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{uuid4().hex}.json"
    record = {
        "_source": ARCHIVE_URL,
        "_location": location,
        "_ingested_at": now.isoformat(),
        "payload": payload,
    }
    path.write_text(json.dumps(record, indent=2))
    return path

## EU Poultry data extractor

In [ ]:
"""Extract: ask the source for data and keep an untouched Bronze copy."""

from datetime import date
from pathlib import Path

BASE_URL = "https://api.tech.ec.europa.eu/agrifood/api/poultry/prices"

# ==============================================================================
# Valid Query Parameter Reference (from OpenAPI Spec)
# ==============================================================================
# memberStateCodes : list[str] -> EU ISO2 country codes (e.g., ["PT", "ES", "FR", "DE"])
# products         : list[str] -> Must be one or more of: "Broiler", "Breast", "Legs"
# priceTypes       : list[str] -> "Selling price" (default if omitted),
#                                 "Retail buying prices", "Non-retail buying prices", "All"
# years            : list[int] -> 4-digit years (e.g., [2024, 2025])
# months           : list[int] -> Month numbers 1-12 (e.g., [1, 6, 12])
# weeks            : list[int] -> Week numbers 1-53 (week 1 = first week of January)
# beginDate        : str       -> Start date in "dd/mm/yyyy" format (e.g., "22/01/2024")
# endDate          : str       -> End date in "dd/mm/yyyy" format (e.g., "22/04/2024")
# ==============================================================================


class TemporarySourceError(RuntimeError):
    """The source failed in a way that might work if we try again."""


def _build_params(
    member_state_codes: list[str] | None = None,  # e.g., ["PT", "ES"]
    products: list[str] | None = None,  # "Broiler", "Breast", "Legs"
    price_types: list[str]
    | None = None,  # "Selling price" (default), "Retail buying prices", "Non-retail buying prices", "All"
    years: list[int] | None = None,  # 4-digit years, e.g., [2024, 2025]
    months: list[int] | None = None,  # 1 to 12
    weeks: list[int] | None = None,  # 1 to 53 (week 1 = first week of Jan)
    begin_date: str | None = None,  # "dd/mm/yyyy"
    end_date: str | None = None,  # "dd/mm/yyyy"
) -> dict[str, str]:
    """Build query parameters with comma-separated arrays (explode: false)."""
    params: dict[str, str] = {}

    if member_state_codes:
        params["memberStateCodes"] = ",".join(member_state_codes)
    if products:
        params["products"] = ",".join(products)
    if price_types:
        params["priceTypes"] = ",".join(price_types)
    if years:
        params["years"] = ",".join(map(str, years))
    if months:
        params["months"] = ",".join(map(str, months))
    if weeks:
        params["weeks"] = ",".join(map(str, weeks))
    if begin_date:
        params["beginDate"] = begin_date
    if end_date:
        params["endDate"] = end_date

    return params


def _handle_response(response: httpx.Response) -> list[dict]:
    """Validate response status and parse JSON payload."""
    if response.status_code == 429 or response.status_code >= 500:
        raise TemporarySourceError(f"EU Agri-food API answered {response.status_code}. Try later.")

    # API returns 404 when valid query filters yield zero matching records
    if response.status_code == 404:
        return []

    response.raise_for_status()
    return response.json()


# --- Synchronous Client ---
def get_poultry_prices(
    member_state_codes: list[str] | None = None,  # EU ISO2 codes (e.g., ["PT", "ES"])
    products: list[str] | None = None,  # "Broiler", "Breast", "Legs"
    price_types: list[str]
    | None = None,  # "Selling price" (default), "Retail buying prices", "Non-retail buying prices", "All"
    years: list[int] | None = None,  # 4-digit years (e.g., [2024])
    months: list[int] | None = None,  # 1-12
    weeks: list[int] | None = None,  # 1-53 (week 1 = first week of Jan)
    begin_date: str | None = None,  # "dd/mm/yyyy"
    end_date: str | None = None,  # "dd/mm/yyyy"
) -> list[dict]:
    """Download weekly poultry prices from the EU Agri-food Data Portal API.

    Raises:
        TemporarySourceError: Network trouble, rate limiting (429), or server error (5xx).
        httpx.HTTPStatusError: Any other bad HTTP response (e.g., 400 Bad Request).
    """
    params = _build_params(
        member_state_codes,
        products,
        price_types,
        years,
        months,
        weeks,
        begin_date,
        end_date,
    )

    with httpx.Client(headers={"Accept": "application/json"}, timeout=30.0) as client:
        try:
            response = client.get(BASE_URL, params=params)
        except httpx.TransportError as error:
            raise TemporarySourceError(f"Could not reach EU Agri-food API: {error}") from error

        return _handle_response(response)

In [ ]:
try:
    data = get_poultry_prices(
        member_state_codes=["PT", "ES"],
        products=["Broiler"],
        price_types=["Selling price"],
        begin_date="01/01/2024",
        end_date="31/12/2024",
    )
    print(f"Retrieved {len(data)} records.")
    for row in data[:5]:
        print(row)
except TemporarySourceError as err:
    # Safe to retry later (e.g., with exponential backoff)
    print(f"Temporary failure: {err}")
except httpx.HTTPStatusError as err:
    # Permanent client error (e.g., invalid query syntax) — do not retry blindly
    print(f"Request failed with status {err.response.status_code}: {err}")

In [ ]:
def save_bronze(
    payload: list[dict],
    bronze_dir: str | Path,
    member_state_codes: list[str] | None = None,
    query_params: dict[str, str] | None = None,
) -> Path:
    """Save the raw API response list exactly as received, plus ingestion metadata."""
    now = datetime.now(UTC)
    member_state_partition = (
        "-".join(sorted(code.upper() for code in member_state_codes))
        if member_state_codes
        else "ALL"
    )

    folder = (
        Path(bronze_dir) / f"ingest_date={now.date()}" / f"member_state={member_state_partition}"
    )
    folder.mkdir(parents=True, exist_ok=True)

    path = folder / f"{uuid4().hex}.json"
    record = {
        "_source": BASE_URL,
        "_member_state": member_state_partition,
        "_query_params": query_params or {},
        "_record_count": len(payload),
        "_ingested_at": now.isoformat(),
        "payload": payload,
    }
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    return path

In [ ]:
states = ["PT", "ES"]
try:
    payload = get_poultry_prices(
        member_state_codes=states,
        products=["Broiler"],
        price_types=["Selling price"],
        begin_date="01/01/2024",
        end_date="31/12/2024",
    )
    saved_path = save_bronze(
        payload=payload,
        bronze_dir="data/bronze/poultry_prices",
        member_state_codes=states,
        query_params=_build_params(
            member_state_codes=states,
            products=["Broiler"],
            price_types=["Selling price"],
            begin_date="01/01/2024",
            end_date="31/12/2024",
        ),
    )
    print(f"Saved {len(payload)} records to {saved_path}")
except TemporarySourceError as err:
    print(f"Temporary failure: {err}")
except httpx.HTTPStatusError as err:
    print(f"Request failed with status {err.response.status_code}: {err}")

In [ ]:
from pathlib import Path

import httpx

BASE_URL = "https://api.tech.ec.europa.eu/agrifood/api/poultry/prices"

# ==============================================================================
# Valid Query Parameter Reference (from OpenAPI Spec)
# ==============================================================================
# memberStateCodes : list[str] -> EU ISO2 country codes (e.g., ["PT", "ES", "FR", "DE"])
# products         : list[str] -> Must be one or more of: "Broiler", "Breast", "Legs"
# priceTypes       : list[str] -> "Selling price" (default if omitted),
#                                 "Retail buying prices", "Non-retail buying prices", "All"
# years            : list[int] -> 4-digit years (e.g., [2024, 2025])
# months           : list[int] -> Month numbers 1-12 (e.g., [1, 6, 12])
# weeks            : list[int] -> Week numbers 1-53 (week 1 = first week of January)
# beginDate        : str       -> Start date in "dd/mm/yyyy" format (e.g., "22/01/2024")
# endDate          : str       -> End date in "dd/mm/yyyy" format (e.g., "22/04/2024")
# ==============================================================================


class TemporarySourceError(RuntimeError):
    """The source failed in a way that might work if we try again."""


def _build_params(
    member_state_codes: list[str] | None = None,  # e.g., ["PT", "ES"]
    products: list[str] | None = None,  # "Broiler", "Breast", "Legs"
    price_types: list[str]
    | None = None,  # "Selling price" (default), "Retail buying prices", "Non-retail buying prices", "All"
    years: list[int] | None = None,  # 4-digit years, e.g., [2024, 2025]
    months: list[int] | None = None,  # 1 to 12
    weeks: list[int] | None = None,  # 1 to 53 (week 1 = first week of Jan)
    begin_date: str | None = None,  # "dd/mm/yyyy"
    end_date: str | None = None,  # "dd/mm/yyyy"
) -> dict[str, str]:
    """Build query parameters with comma-separated arrays (explode: false)."""
    params: dict[str, str] = {}

    if member_state_codes:
        params["memberStateCodes"] = ",".join(member_state_codes)
    if products:
        params["products"] = ",".join(products)
    if price_types:
        params["priceTypes"] = ",".join(price_types)
    if years:
        params["years"] = ",".join(map(str, years))
    if months:
        params["months"] = ",".join(map(str, months))
    if weeks:
        params["weeks"] = ",".join(map(str, weeks))
    if begin_date:
        params["beginDate"] = begin_date
    if end_date:
        params["endDate"] = end_date

    return params


def get_poultry_prices(
    member_state_codes: list[str] | None = None,  # EU ISO2 codes (e.g., ["PT", "ES"])
    products: list[str] | None = None,  # "Broiler", "Breast", "Legs"
    price_types: list[str]
    | None = None,  # "Selling price" (default), "Retail buying prices", "Non-retail buying prices", "All"
    years: list[int] | None = None,  # 4-digit years (e.g., [2024])
    months: list[int] | None = None,  # 1-12
    weeks: list[int] | None = None,  # 1-53 (week 1 = first week of Jan)
    begin_date: str | None = None,  # "dd/mm/yyyy"
    end_date: str | None = None,  # "dd/mm/yyyy"
) -> list[dict]:
    """Download weekly poultry prices from the EU Agri-food Data Portal API.

    Raises:
        TemporarySourceError: Network trouble, rate limiting (429), or server error (5xx).
        httpx.HTTPStatusError: Any other bad HTTP response (e.g., 400 Bad Request).
    """
    params = _build_params(
        member_state_codes,
        products,
        price_types,
        years,
        months,
        weeks,
        begin_date,
        end_date,
    )

    with httpx.Client(headers={"Accept": "application/json"}, timeout=30.0) as client:
        try:
            response = client.get(BASE_URL, params=params)
        except httpx.TransportError as error:
            raise TemporarySourceError(f"Could not reach EU Agri-food API: {error}") from error

        if response.status_code == 429 or response.status_code >= 500:
            raise TemporarySourceError(
                f"EU Agri-food API answered {response.status_code}. Try later."
            )

        # API returns 404 when valid query filters yield zero matching records
        if response.status_code == 404:
            return []

        response.raise_for_status()
        return response.json()


def save_bronze(
    payload: list[dict],
    bronze_dir: str | Path,
    member_state_codes: list[str] | None = None,
    query_params: dict[str, str] | None = None,
) -> Path:
    """Save the raw API response list exactly as received, plus ingestion metadata."""
    now = datetime.now(UTC)
    member_state_partition = (
        "-".join(sorted(code.upper() for code in member_state_codes))
        if member_state_codes
        else "ALL"
    )

    folder = (
        Path(bronze_dir) / f"ingest_date={now.date()}" / f"member_state={member_state_partition}"
    )
    folder.mkdir(parents=True, exist_ok=True)

    path = folder / f"{uuid4().hex}.json"
    record = {
        "_source": BASE_URL,
        "_member_state": member_state_partition,
        "_query_params": query_params or {},
        "_record_count": len(payload),
        "_ingested_at": now.isoformat(),
        "payload": payload,
    }
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    return path

In [ ]:
states = ["PT", "ES"]

payload = get_poultry_prices(
    member_state_codes=states,
    products=["Broiler"],
    price_types=["Selling price"],
    begin_date="01/01/2024",
    end_date="31/12/2024",
)
saved_path = save_bronze(
    payload=payload,
    bronze_dir="data/bronze/poultry_prices",
    member_state_codes=states,
    query_params=_build_params(
        member_state_codes=states,
        products=["Broiler"],
        price_types=["Selling price"],
        begin_date="01/01/2024",
        end_date="31/12/2024",
    ),
)

print(f"Saved {len(payload)} records to {saved_path}")

In [ ]:
from datetime import date, timedelta
from pathlib import Path

import httpx

try:
    from .config import (
        BASE_URL,
        BRONZE_DIR,
        FIRST_LOAD_DAYS,
        HTTP_TIMEOUT,
        MEMBER_STATE_CODES,
        PRICE_TYPES,
        PRODUCTS,
    )
except ImportError:
    pass
    # from config import (
    #     BASE_URL,
    #     BRONZE_DIR,
    #     FIRST_LOAD_DAYS,
    #     HTTP_TIMEOUT,
    #     MEMBER_STATE_CODES,
    #     PRICE_TYPES,
    #     PRODUCTS,
    # )

# ==============================================================================
# Valid Query Parameter Reference (from OpenAPI Spec)
# ==============================================================================
# memberStateCodes : list[str]   -> EU ISO2 country codes (e.g., ["PT", "ES", "FR", "DE"])
# products         : list[str]   -> Must be one or more of: "Broiler", "Breast", "Legs"
# priceTypes       : list[str]   -> "Selling price" (default if omitted),
#                                   "Retail buying prices", "Non-retail buying prices", "All"
# years            : list[int]   -> 4-digit years (e.g., [2024, 2025])
# months           : list[int]   -> Month numbers 1-12 (e.g., [1, 6, 12])
# weeks            : list[int]   -> Week numbers 1-53 (week 1 = first week of January)
# beginDate        : date | str  -> Start date as a date object or "dd/mm/yyyy" string
# endDate          : date | str  -> End date as a date object or "dd/mm/yyyy" string
# ==============================================================================


class TemporarySourceError(RuntimeError):
    """The source failed in a way that might work if we try again."""


def _format_api_date(value: date | str) -> str:
    """Convert Python date/datetime objects to the 'dd/mm/yyyy' format required by the API."""
    if isinstance(value, date):
        return value.strftime("%d/%m/%Y")
    return value


def build_params(
    member_state_codes: list[str] | None = None,  # e.g., ["PT", "ES"]
    products: list[str] | None = None,  # "Broiler", "Breast", "Legs"
    price_types: list[str]
    | None = None,  # "Selling price" (default), "Retail buying prices", "Non-retail buying prices", "All"
    years: list[int] | None = None,  # 4-digit years, e.g., [2024, 2025]
    months: list[int] | None = None,  # 1 to 12
    weeks: list[int] | None = None,  # 1 to 53 (week 1 = first week of Jan)
    begin_date: date | str | None = None,  # date object or "dd/mm/yyyy"
    end_date: date | str | None = None,  # date object or "dd/mm/yyyy"
) -> dict[str, str]:
    """Build query parameters with comma-separated arrays (explode: false)."""
    params: dict[str, str] = {}

    if member_state_codes:
        params["memberStateCodes"] = ",".join(member_state_codes)
    if products:
        params["products"] = ",".join(products)
    if price_types:
        params["priceTypes"] = ",".join(price_types)
    if years:
        params["years"] = ",".join(map(str, years))
    if months:
        params["months"] = ",".join(map(str, months))
    if weeks:
        params["weeks"] = ",".join(map(str, weeks))
    if begin_date:
        params["beginDate"] = _format_api_date(begin_date)
    if end_date:
        params["endDate"] = _format_api_date(end_date)

    return params


def get_poultry_prices(
    member_state_codes: list[str] | None = MEMBER_STATE_CODES,
    products: list[str] | None = PRODUCTS,
    price_types: list[str] | None = PRICE_TYPES,
    years: list[int] | None = None,
    months: list[int] | None = None,
    weeks: list[int] | None = None,
    begin_date: date | str | None = None,
    end_date: date | str | None = None,
) -> list[dict]:
    """Download weekly poultry prices from the EU Agri-food Data Portal API.

    Raises:
        TemporarySourceError: Network trouble, rate limiting (429), or server error (5xx).
        httpx.HTTPStatusError: Any other bad HTTP response (e.g., 400 Bad Request).
    """
    params = build_params(
        member_state_codes=member_state_codes,
        products=products,
        price_types=price_types,
        years=years,
        months=months,
        weeks=weeks,
        begin_date=begin_date,
        end_date=end_date,
    )

    with httpx.Client(headers={"Accept": "application/json"}, timeout=HTTP_TIMEOUT) as client:
        try:
            response = client.get(BASE_URL, params=params)
        except httpx.TransportError as error:
            raise TemporarySourceError(f"Could not reach EU Agri-food API: {error}") from error

        if response.status_code == 429 or response.status_code >= 500:
            raise TemporarySourceError(
                f"EU Agri-food API answered {response.status_code}. Try later."
            )

        # API returns 404 when valid query filters yield zero matching records
        if response.status_code == 404:
            return []

        response.raise_for_status()
        return response.json()


def save_bronze(
    payload: list[dict],
    bronze_dir: str | Path = BRONZE_DIR,
    member_state_codes: list[str] | None = MEMBER_STATE_CODES,
    products: list[str] | None = PRODUCTS,
    price_types: list[str] | None = PRICE_TYPES,
    years: list[int] | None = None,
    months: list[int] | None = None,
    weeks: list[int] | None = None,
    begin_date: date | str | None = None,
    end_date: date | str | None = None,
    query_params: dict[str, str] | None = None,
) -> Path:
    """Save the raw API response list exactly as received, plus ingestion metadata."""
    now = datetime.now(UTC)
    member_state_partition = (
        "-".join(sorted(code.upper() for code in member_state_codes))
        if member_state_codes
        else "ALL"
    )

    resolved_params = (
        query_params
        if query_params is not None
        else build_params(
            member_state_codes=member_state_codes,
            products=products,
            price_types=price_types,
            years=years,
            months=months,
            weeks=weeks,
            begin_date=begin_date,
            end_date=end_date,
        )
    )

    folder = (
        Path(bronze_dir) / f"ingest_date={now.date()}" / f"member_state={member_state_partition}"
    )
    folder.mkdir(parents=True, exist_ok=True)

    path = folder / f"{uuid4().hex}.json"
    record = {
        "_source": BASE_URL,
        "_member_state": member_state_partition,
        "_query_params": resolved_params,
        "_record_count": len(payload),
        "_ingested_at": now.isoformat(),
        "payload": payload,
    }
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    return path

In [ ]:
if __name__ == "__main__":
    end = datetime.now(UTC).date()
    start = end - timedelta(days=FIRST_LOAD_DAYS)

    payload = get_poultry_prices(begin_date=start, end_date=end)
    saved_path = save_bronze(payload=payload, begin_date=start, end_date=end)
    print(f"Saved {len(payload)} records to {saved_path}")

In [5]:
"""Extract: ask the source for data and keep an untouched Bronze copy."""

from datetime import UTC, date, datetime
from pathlib import Path

import httpx

BASE_URL = "https://api.tech.ec.europa.eu/agrifood/api/poultry/prices"

# ==============================================================================
# Valid Query Parameter Reference (from OpenAPI Spec)
# ==============================================================================
# memberStateCodes : list[str]   -> EU ISO2 country codes (e.g., ["PT", "ES", "FR", "DE"])
# products         : list[str]   -> Must be one or more of: "Broiler", "Breast", "Legs"
# priceTypes       : list[str]   -> "Selling price" (default if omitted),
#                                   "Retail buying prices", "Non-retail buying prices", "All"
# years            : list[int]   -> 4-digit years (e.g., [2024, 2025])
# months           : list[int]   -> Month numbers 1-12 (e.g., [1, 6, 12])
# weeks            : list[int]   -> Week numbers 1-53 (week 1 = first week of January)
# beginDate        : date | str  -> Start date as a date object or "dd/mm/yyyy" string
# endDate          : date | str  -> End date as a date object or "dd/mm/yyyy" string
# ==============================================================================


class TemporarySourceError(RuntimeError):
    """The source failed in a way that might work if we try again."""


def get_poultry_prices(
    member_state_codes: list[str] | None = None,
    products: list[str] | None = None,
    price_types: list[str] | None = None,
    years: list[int] | None = None,
    months: list[int] | None = None,
    weeks: list[int] | None = None,
    begin_date: date | str | None = None,
    end_date: date | str | None = None,
    base_url: str = BASE_URL,
    timeout: float = 30.0,
) -> list[dict]:
    """Download weekly poultry prices from the EU Agri-food Data Portal API.

    Raises:
        TemporarySourceError: Network trouble, rate limiting (429), or server error (5xx).
        httpx.HTTPStatusError: Any other bad HTTP response (e.g., 400 Bad Request).
    """
    params: dict[str, str] = {}

    if member_state_codes:
        params["memberStateCodes"] = ",".join(member_state_codes)
    if products:
        params["products"] = ",".join(products)
    if price_types:
        params["priceTypes"] = ",".join(price_types)
    if years:
        params["years"] = ",".join(map(str, years))
    if months:
        params["months"] = ",".join(map(str, months))
    if weeks:
        params["weeks"] = ",".join(map(str, weeks))
    if begin_date:
        params["beginDate"] = (
            begin_date.strftime("%d/%m/%Y") if isinstance(begin_date, date) else begin_date
        )
    if end_date:
        params["endDate"] = (
            end_date.strftime("%d/%m/%Y") if isinstance(end_date, date) else end_date
        )

    with httpx.Client(headers={"Accept": "application/json"}, timeout=timeout) as client:
        try:
            response = client.get(base_url, params=params)
        except httpx.TransportError as error:
            raise TemporarySourceError(f"Could not reach EU Agri-food API: {error}") from error

        if response.status_code == 429 or response.status_code >= 500:
            raise TemporarySourceError(
                f"EU Agri-food API answered {response.status_code}. Try later."
            )

        # API returns 404 when valid query filters yield zero matching records
        if response.status_code == 404:
            return []

        response.raise_for_status()
        return response.json()


def save_bronze(
    payload: list[dict],
    bronze_dir: str | Path,
    *,
    member_state_codes: list[str] | None = None,
    source_url: str = BASE_URL,
) -> Path:
    """Save the raw API response list exactly as received, plus ingestion metadata."""
    now = datetime.now(UTC)
    member_state_partition = (
        "-".join(sorted(code.upper() for code in member_state_codes))
        if member_state_codes
        else "ALL"
    )

    folder = (
        Path(bronze_dir)
        / "poultry_prices"
        / f"ingest_date={now.date()}"
        / f"member_state={member_state_partition}"
    )
    folder.mkdir(parents=True, exist_ok=True)

    path = folder / f"{uuid4().hex}.json"
    record = {
        "_source": source_url,
        "_member_state": member_state_partition,
        "_record_count": len(payload),
        "_ingested_at": now.isoformat(),
        "payload": payload,
    }
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    return path

In [6]:
from datetime import UTC, datetime, timedelta

BASE_URL = "https://api.tech.ec.europa.eu/agrifood/api/poultry/prices"
BRONZE_DIR = "data/bronze"
HTTP_TIMEOUT = 30.0
FIRST_LOAD_DAYS = 365

MEMBER_STATE_CODES = None
PRODUCTS = None
PRICE_TYPES = ["Selling price"]

end_date = datetime.now(UTC).date()
begin_date = end_date - timedelta(days=FIRST_LOAD_DAYS)

payload = get_poultry_prices(
    member_state_codes=MEMBER_STATE_CODES,
    products=PRODUCTS,
    price_types=PRICE_TYPES,
    begin_date=begin_date,
    end_date=end_date,
    base_url=BASE_URL,
    timeout=HTTP_TIMEOUT,
)
saved_path = save_bronze(
    payload,
    BRONZE_DIR,
    member_state_codes=MEMBER_STATE_CODES,
    source_url=BASE_URL,
)

In [7]:
print(f"Saved {len(payload)} records to {saved_path}")

Saved 3101 records to data/bronze/poultry_prices/ingest_date=2026-11-27/member_state=ALL/5f6f4cce541a4999b8bb1e7446c80a8c.json
